# Hospital Readmission Prediction: Data Understanding

## Project objective

This project examines which patient, hospitalization, and treatment characteristics are associated with readmission within 30 days among hospital encounters involving patients with diabetes.

The analysis is an educational portfolio project and is not intended for clinical decision-making.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

print("Python environment is ready.")
print("pandas version:", pd.__version__)

Python environment is ready.
pandas version: 3.0.5


In [3]:
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "diabetic_data.csv"
mapping_path = project_root / "data" / "raw" / "IDS_mapping.csv"

print("Dataset path:", data_path)
print("Dataset found:", data_path.exists())
print("Mapping file found:", mapping_path.exists())

Dataset path: c:\Users\ezont\Desktop\PersonalProj\hospital_readmission_prediction\data\raw\diabetic_data.csv
Dataset found: True
Mapping file found: True


In [4]:
# Load the raw files without changing their og values
encounters_raw = pd.read_csv(data_path, low_memory=False)
id_mapping_raw = pd.read_csv(mapping_path)

print("Hospital encounters:", encounters_raw.shape)
print("Mapping table:", id_mapping_raw.shape)

encounters_raw.head()

Hospital encounters: (101766, 50)
Mapping table: (67, 2)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
print("Column names:")
for number, column in enumerate(encounters_raw.columns, start=1):
    print(f"{number:>2}. {column}")

Column names:
 1. encounter_id
 2. patient_nbr
 3. race
 4. gender
 5. age
 6. weight
 7. admission_type_id
 8. discharge_disposition_id
 9. admission_source_id
10. time_in_hospital
11. payer_code
12. medical_specialty
13. num_lab_procedures
14. num_procedures
15. num_medications
16. number_outpatient
17. number_emergency
18. number_inpatient
19. diag_1
20. diag_2
21. diag_3
22. number_diagnoses
23. max_glu_serum
24. A1Cresult
25. metformin
26. repaglinide
27. nateglinide
28. chlorpropamide
29. glimepiride
30. acetohexamide
31. glipizide
32. glyburide
33. tolbutamide
34. pioglitazone
35. rosiglitazone
36. acarbose
37. miglitol
38. troglitazone
39. tolazamide
40. examide
41. citoglipton
42. insulin
43. glyburide-metformin
44. glipizide-metformin
45. glimepiride-pioglitazone
46. metformin-rosiglitazone
47. metformin-pioglitazone
48. change
49. diabetesMed
50. readmitted


In [6]:
# Basic dataset summary
summary = pd.DataFrame({
    "data_type": encounters_raw.dtypes.astype(str),
    "unique_values": encounters_raw.nunique(),
    "question_mark_count": (encounters_raw == "?").sum(),
    "blank_count": encounters_raw.isna().sum()
})

summary.sort_values(
    by="question_mark_count",
    ascending=False
).head(15)

,data_type,unique_values,question_mark_count,blank_count
weight,str,10,98569,0
medical_specialty,str,73,49949,0
payer_code,str,18,40256,0
race,str,6,2273,0
diag_3,str,790,1423,0
diag_2,str,749,358,0
diag_1,str,717,21,0
admission_type_id,int64,8,0,0
patient_nbr,int64,71518,0,0
encounter_id,int64,101766,0,0


## Readmission outcome

The original `readmitted` variable has three categories:

- `<30`: The encounter was followed by another inpatient admission within 30 days.
- `>30`: A subsequent inpatient admission occurred after 30 days.
- `NO`: No subsequent inpatient admission was recorded.

For this project, `<30` is the positive class. The other two categories form the negative class.

In [7]:
readmission_counts = (
    encounters_raw["readmitted"]
    .value_counts()
    .rename_axis("original_outcome")
    .reset_index(name="encounters")
)

readmission_counts["percentage"] = (
    readmission_counts["encounters"]
    / len(encounters_raw)
    * 100
).round(2)

readmission_counts

,original_outcome,encounters,percentage
0,NO,54864,53.91
1,>30,35545,34.93
2,<30,11357,11.16


In [8]:
encounters_raw["readmitted_30d"] = (
    encounters_raw["readmitted"] == "<30"
).astype(int)

binary_target_summary = (
    encounters_raw["readmitted_30d"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Not readmitted within 30 days",
        1: "Readmitted within 30 days"
    })
    .rename_axis("outcome")
    .reset_index(name="encounters")
)

binary_target_summary["percentage"] = (
    binary_target_summary["encounters"]
    / len(encounters_raw)
    * 100
).round(2)

binary_target_summary

,outcome,encounters,percentage
0,Not readmitted within 30 days,90409,88.84
1,Readmitted within 30 days,11357,11.16


In [9]:
patient_encounter_counts = encounters_raw["patient_nbr"].value_counts()

patient_summary = pd.Series({
    "Total encounters": len(encounters_raw),
    "Unique encounter IDs": encounters_raw["encounter_id"].nunique(),
    "Unique patients": encounters_raw["patient_nbr"].nunique(),
    "Patients with multiple encounters": (patient_encounter_counts > 1).sum(),
    "Maximum encounters for one patient": patient_encounter_counts.max()
})

patient_summary

Total encounters                      101766
Unique encounter IDs                  101766
Unique patients                        71518
Patients with multiple encounters      16773
Maximum encounters for one patient        40
dtype: int64

### Patient-level data leakage risk

The dataset contains repeated encounters for some patients. A random encounter-level split could place records belonging to the same patient in both the training and test sets, allowing patient-specific information to leak between them. Therefore, model development will use a patient-level group split based on `patient_nbr`.

In [10]:
pd.set_option("display.max_rows", 100)

id_mapping_raw

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


In [11]:
coded_fields = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for column in coded_fields:
    print(f"\n{column}")
    print(encounters_raw[column].value_counts().sort_index())


admission_type_id
admission_type_id
1    53990
2    18480
3    18869
4       10
5     4785
6     5291
7       21
8      320
Name: count, dtype: int64

discharge_disposition_id
discharge_disposition_id
1     60234
2      2128
3     13954
4       815
5      1184
6     12902
7       623
8       108
9        21
10        6
11     1642
12        3
13      399
14      372
15       63
16       11
17       14
18     3691
19        8
20        2
22     1993
23      412
24       48
25      989
27        5
28      139
Name: count, dtype: int64

admission_source_id
admission_source_id
1     29565
2      1104
3       187
4      3187
5       855
6      2264
7     57494
8        16
9       125
10        8
11        2
13        1
14        2
17     6781
20      161
22       12
25        2
Name: count, dtype: int64


In [12]:
question_mark_counts = (encounters_raw == "?").sum()

missing_summary = pd.DataFrame({
    "missing_count": question_mark_counts,
    "missing_percentage": (
        question_mark_counts / len(encounters_raw) * 100
    ).round(2)
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

missing_summary

,missing_count,missing_percentage
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


### Missing-value observations

The raw dataset represents some missing values using `?` rather than standard null values. These values must be converted to missing values during data cleaning.

Variables with substantial missingness require special consideration. Columns should not be removed solely because they contain missing data; their relevance, reliability, and potential availability at prediction time must also be evaluated.

In [13]:
numeric_fields = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

encounters_raw[numeric_fields].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
time_in_hospital,101766.0,4.40,2.99,1.0,2.0,4.0,6.0,14.0
num_lab_procedures,101766.0,43.10,19.67,1.0,31.0,44.0,57.0,132.0
num_procedures,101766.0,1.34,1.71,0.0,0.0,1.0,2.0,6.0
num_medications,101766.0,16.02,8.13,1.0,10.0,15.0,20.0,81.0
number_outpatient,101766.0,0.37,1.27,0.0,0.0,0.0,0.0,42.0
number_emergency,101766.0,0.20,0.93,0.0,0.0,0.0,0.0,76.0
number_inpatient,101766.0,0.64,1.26,0.0,0.0,0.0,1.0,21.0
number_diagnoses,101766.0,7.42,1.93,1.0,6.0,8.0,9.0,16.0


In [14]:
categorical_fields = [
    "race",
    "gender",
    "age",
    "A1Cresult",
    "max_glu_serum",
    "change",
    "diabetesMed",
    "readmitted"
]

for column in categorical_fields:
    print(f"\n--- {column} ---")
    print(encounters_raw[column].value_counts(dropna=False))


--- race ---
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

--- gender ---
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

--- age ---
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

--- A1Cresult ---
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

--- max_glu_serum ---
max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64

--- change ---
change
No    54755
Ch    47011
Name: count, dtype: int64

--- diabetesMed ---
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64

--- readmitted ---
readmitted
NO     54864
>30    35545
<30    11357


In [15]:
identifier_checks = pd.Series({
    "Encounter IDs are unique":
        encounters_raw["encounter_id"].is_unique,

    "Number of unique encounters":
        encounters_raw["encounter_id"].nunique(),

    "Number of unique patients":
        encounters_raw["patient_nbr"].nunique(),

    "Repeated encounter IDs":
        encounters_raw["encounter_id"].duplicated().sum(),

    "Repeated patient rows beyond first encounter":
        encounters_raw["patient_nbr"].duplicated().sum()
})

identifier_checks

Encounter IDs are unique                          True
Number of unique encounters                     101766
Number of unique patients                        71518
Repeated encounter IDs                               0
Repeated patient rows beyond first encounter     30248
dtype: object

## Initial candidate features

The initial modeling features will represent information about the patient, hospitalization, prior healthcare utilization, diagnoses, testing, and treatment.

Candidate feature groups include:

- **Demographics:** race, gender, and age
- **Hospital encounter:** admission type, discharge disposition, admission source, and time in hospital
- **Clinical activity:** laboratory procedures, other procedures, medications, and diagnoses
- **Previous utilization:** outpatient, emergency, and inpatient visits during the preceding year
- **Testing:** HbA1c result and maximum serum glucose result
- **Treatment:** medication changes, diabetes-medication use, and individual diabetes medications
- **Diagnoses:** primary, secondary, and additional diagnosis categories

Identifiers such as `encounter_id` and `patient_nbr` will not be used as predictive features. The patient identifier will be retained only to create leakage-safe training and test groups.

## Initial data-quality findings

- The dataset contains 101,766 hospital encounters and 50 original variables.
- Each `encounter_id` identifies one unique hospital encounter.
- The data include 71,518 unique patients.
- Some patients have multiple hospital encounters, with a maximum of 40 encounters for one patient.
- Missing values are represented by `?` in several categorical columns.
- The hospital admission, discharge, and admission-source variables use numeric identifiers that require lookup-table descriptions.
- The positive outcome is uncommon: 11,357 encounters, or 11.16%, were followed by readmission within 30 days.
- Patient-level splitting will be required to prevent data leakage.
- Accuracy alone will not adequately evaluate the model because of the target imbalance.